# MannuMitra AI-Based Predictive Farming System
## 30-Crop Random Forest Model: Final Rigorous Validation & Agronomic Audit

**Objective:** Complete an independent, end-to-end validation of the experimental 30-crop Random Forest model, evaluating data quality, agronomic consistency, reproducibility, and classification performance across Karnataka's primary commercial, plantation, cereal, and pulse crops.

### Protected Production Baseline
- Production model (`models/baseline/crop_random_forest/crop_model.pkl`) is completely isolated and unchanged.
- Production model SHA-256: `0f0e9a2e2ae1ebe19fc861f64fc8a737534a32de89143747d04e5e456deac779`.

### Key Experimental Constraints
1. **Zero Data Leakage:** Stratified 80/20 train/test split. All model selection, feature inspection, and hyperparameter tuning are conducted strictly on the training partition via 5-fold cross-validation.
2. **Held-Out Test Oracle:** The test set is evaluated exactly once for the final selected model.
3. **Transparent Provenance:** All records are documented as synthetic/simulated. ICAR-CPCRI parameters represent agronomic ratio calibrations, not empirical field trials.

In [ ]:
import os
import json
import time
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split

SEED = 42
np.random.seed(SEED)
print(f"Random seed: {SEED}")

### 1. Dataset Loading and SHA-256 Verification
We load `karnataka_30_crops_cpcri_rainfall_fixed.csv`, verify its cryptographic checksum, and inspect class distribution.

In [ ]:
from pathlib import Path
ROOT_DIR = Path('..')
DATASET_PATH = ROOT_DIR / 'data' / 'rf_30_class' / 'karnataka_30_crops_cpcri_rainfall_fixed.csv'
if not DATASET_PATH.exists():
    DATASET_PATH = ROOT_DIR / 'data' / 'processed' / 'karnataka_30_crops_cpcri_rainfall_fixed.csv'
with open(DATASET_PATH, 'rb') as f:
    dataset_sha256 = hashlib.sha256(f.read()).hexdigest().upper()
print(f'Dataset Path:   {DATASET_PATH}')
print(f'Dataset SHA256: {dataset_sha256}')
df = pd.read_csv(DATASET_PATH)
print(f'Loaded {len(df)} rows across {df["label"].nunique()} crop classes.')
df.head(5)


### 2. Stratified Train / Test Partitioning
We split the dataset into 80% training (15,024 samples) and 20% held-out test (3,756 samples), stratified by crop label.

In [ ]:
FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
TARGET = 'label'

X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print(f"Training set samples: {len(X_train)} (80%)")
print(f"Held-out test samples: {len(X_test)} (20%)")

### 3. Model Training & 5-Fold Stratified Cross-Validation on Training Set
Cross-validation is strictly isolated to `X_train` and `y_train` to prevent test-set leakage.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {'accuracy': 'accuracy', 'balanced_accuracy': 'balanced_accuracy', 'f1_macro': 'f1_macro', 'f1_weighted': 'f1_weighted'}

cv_scores = cross_validate(rf_model, X_train, y_train, cv=skf, scoring=scoring, n_jobs=1)
print(f"5-Fold CV Accuracy:          {np.mean(cv_scores['test_accuracy'])*100:.2f}% ± {np.std(cv_scores['test_accuracy'])*100:.2f}%")
print(f"5-Fold CV Balanced Accuracy: {np.mean(cv_scores['test_balanced_accuracy'])*100:.2f}% ± {np.std(cv_scores['test_balanced_accuracy'])*100:.2f}%")
print(f"5-Fold CV Macro F1:          {np.mean(cv_scores['test_f1_macro'])*100:.2f}% ± {np.std(cv_scores['test_f1_macro'])*100:.2f}%")
print(f"5-Fold CV Weighted F1:       {np.mean(cv_scores['test_f1_weighted'])*100:.2f}% ± {np.std(cv_scores['test_f1_weighted'])*100:.2f}%")

### 4. Final Training and Held-Out Test Evaluation
The selected model is fitted on the entire training set (15,024 samples) and evaluated on `X_test` (3,756 samples).

In [ ]:
rf_model.fit(X_train, y_train)

# Training metrics
y_train_pred = rf_model.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train, y_train_pred)*100:.2f}%")
print(f"Training Macro F1: {f1_score(y_train, y_train_pred, average='macro')*100:.2f}%")

# Held-out Test metrics
y_test_pred = rf_model.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)
test_bal = balanced_accuracy_score(y_test, y_test_pred)
test_f1m = f1_score(y_test, y_test_pred, average='macro')
test_f1w = f1_score(y_test, y_test_pred, average='weighted')

print(f"\nHELD-OUT TEST RESULTS:")
print(f"Test Accuracy:          {test_acc*100:.2f}%")
print(f"Test Balanced Accuracy: {test_bal*100:.2f}%")
print(f"Test Macro F1:          {test_f1m*100:.2f}%")
print(f"Test Weighted F1:       {test_f1w*100:.2f}%")

### 5. Detailed Classification Report & Key Focus Crops

In [ ]:
report_dict = classification_report(y_test, y_test_pred, output_dict=True)
focus_crops = ['arecanut', 'coconut', 'maize', 'rice', 'soybean', 'mungbean', 'blackgram', 'coffee', 'cotton']

print(f"{'Crop':15s} | {'Precision':>10} | {'Recall':>10} | {'F1-Score':>10} | {'Support':>8}")
print('-' * 62)
for fc in focus_crops:
    if fc in report_dict:
        p = report_dict[fc]['precision'] * 100
        r = report_dict[fc]['recall'] * 100
        f1 = report_dict[fc]['f1-score'] * 100
        s = report_dict[fc]['support']
        print(f"{fc:15s} | {p:9.2f}% | {r:9.2f}% | {f1:9.2f}% | {s:8d}")

### 6. Confusion Matrix and Error Analysis

In [ ]:
classes = sorted(y.unique())
cm = confusion_matrix(y_test, y_test_pred, labels=classes)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title(f'30-Crop Confusion Matrix (Acc: {test_acc*100:.2f}%, Macro F1: {test_f1m*100:.2f}%)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()